In [ ]:
import time
import numpy as np

from structuralmodule import structuremodule
from adaptivemutation import AdaptiveMutation

from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.core.problem import Problem
from pymoo.optimize import minimize

In [ ]:
## Change accordingly! ##
proteintarget='./ang2.pdb'

In [ ]:
kmer = 4
minlength=10
maxlength=30
population=10
generations = 10
ssthreshold=0.7
nworkers=30
minimizationsteps=100
hdockspacingstep='2.0'
hdockanglestep='30'
allgenerationsfolder='./nsga2autopilot/'

BASES = ["A", "C", "G", "U"]

In [ ]:
class Evolution(Problem):
    def __init__(self, minlength, maxlength, population, ssthreshold, nworkers, minimizationsteps, hdockspacingstep, hdockanglestep, proteintarget, allgenerationsfolder):
        super().__init__(
            n_var=1 + maxlength,                            # 1 for length + maxlength for bases
            n_obj=2,                                        # Two objectives
            n_constr=0,                                     # No constraints
            xl=[minlength] + [0] * maxlength,               # Length min + Base min
            xu=[maxlength] + [len(BASES) - 1] * maxlength   # Length max + Base max
        )
        self.min_seq_length = minlength
        self.max_seq_length = maxlength
        self.gener = 0
        self.peoplebefore = 0
        self.population = population
        self.ssthreshold = ssthreshold
        self.nworkers = nworkers
        self.minimizationsteps = minimizationsteps
        self.hdockspacingstep = hdockspacingstep
        self.hdockanglestep = hdockanglestep
        self.proteintarget = proteintarget
        self.allgenerationsfolder = allgenerationsfolder

    def _evaluate(self, X, out, *args, **kwargs):
        population_size = X.shape[0]
        
        # Extract lengths and sequences
        lengths = X[:, 0].astype(int)
        raw_sequences = X[:, 1:].astype(int)

        decoded_sequences = [
            ''.join([BASES[int(base)] for base in raw_sequences[i, :lengths[i]]])
            for i in range(population_size)
        ]

        # Remove empty or invalid sequences
        decoded_sequences = [seq for seq in decoded_sequences if seq]

        # Structural module call
        populationdf, peoplebefore, mutprobs = structuremodule(
                                                inpsequences=decoded_sequences,
                                                generation=self.gener,
                                                peoplebefore=self.peoplebefore,
                                                initpopulation=self.population,
                                                nu=0.41,
                                                ssthreshold=self.ssthreshold,
                                                nworkers=self.nworkers,
                                                minimizationsteps=self.minimizationsteps,
                                                hdockspacingstep=self.hdockspacingstep,
                                                hdockanglestep=self.hdockanglestep,
                                                proteintarget=self.proteintarget,
                                                allgenerationsfolder=self.allgenerationsfolder
        )

        # Objective evaluation
        f1_values = np.array([
            populationdf.loc[populationdf['seq'] == seq, 'RASP_norm'].iloc[0]
            for seq in decoded_sequences
        ])

        f2_values = np.array([
            populationdf.loc[populationdf['seq'] == seq, 'MMGBSA_norm'].iloc[0]
            for seq in decoded_sequences
        ])

        # Assign the objectives
        out["F"] = np.column_stack((f1_values, f2_values))

        self.gener += 1
        self.peoplebefore = 0#peoplebefore # if update -- messes up with the verbose

In [ ]:
start = time.time()

problem = Evolution(args.minlength, 
                    args.maxlength, 
                    args.population, 
                    args.ssthreshold,
                    args.nworkers, 
                    args.minimizationsteps, 
                    args.hdockspacingstep,
                    args.hdockanglestep, 
                    args.proteintarget, 
                    args.allgenerationsfolder)

algorithm = NSGA2(pop_size=args.population,
                mutation=AdaptiveMutation(directory=args.allgenerationsfolder, 
                                            k=args.kmer, 
                                            minlength=args.minlength, 
                                            maxlength=args.maxlength)
)

res = minimize(
    problem,
    algorithm,
    termination=('n_gen', args.generations),
    seed=1,
    save_history=True,
    verbose=True
)

end = time.time()

print(f"Time passed: {(end - start)/3600:.2f} hours")
print('Done! Bye.')

In [ ]:
from pymoo.util.misc import stack
import matplotlib.pyplot as plt
import numpy as np

generations = list(range(1, len(res.history) + 1))  # List of generations
f1_values = []
f2_values = []

for algo in res.history:
    F = stack(algo.pop.get("F"))        # Extract objective values
    f1_values.append(np.min(F[:, 0]))  # Minimize F1
    f2_values.append(np.min(F[:, 1]))  # Minimize F2

# Plot Learning Curve
plt.figure(figsize=(10, 6))
plt.plot(generations, f1_values, label="Structure Objective", marker="o")
plt.plot(generations, f2_values, label="Binding Objective", marker="s")
plt.xlabel("Generation")
plt.ylabel("Objectives (scores)")
plt.title("NSGA-II Learning Curve")
plt.legend()
plt.grid()
plt.show()